# Data cleaning: 225-dimensional binary firm representation

This notebook implements Sections 6.1 and 6.2 of the project preparation plan using **Python only**. It:

1. reads the two RDS source files directly in Python and exports Python-native Parquet copies;
2. keeps the 45 firm characteristics in wide form;
3. records missingness **before** any replacement;
4. creates four ordered threshold bits plus one missingness bit per characteristic; and
5. saves a validated `uint8` matrix with 225 binary dimensions.

The notebook deliberately does not create SIC dummies, rank-macro interactions, date leads, or train/validation/test splits.

## 0. Python dependencies

Run the following cell once in the notebook kernel. `pyreadr` reads RDS files without calling R, and `pyarrow` writes Parquet files.

In [1]:
%pip install -q pyreadr pyarrow pandas numpy

Note: you may need to restart the kernel to use updated packages.


## 1. Imports and project paths

The path discovery works whether Jupyter starts in the project root or in the `code` directory. Outputs are written to `data/processed`.

In [2]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import pyreadr

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

def find_project_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    for candidate in candidates:
        data_dir = candidate / "data"
        if (data_dir / "Q_Total_Data84.rds").exists() and (data_dir / "Macro_vars.rds").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find data/Q_Total_Data84.rds and data/Macro_vars.rds "
        "from the current working directory or its parents."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_DIR / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIRM_RDS = DATA_DIR / "Q_Total_Data84.rds"
MACRO_RDS = DATA_DIR / "Macro_vars.rds"

print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

Project root: C:\Users\14282\Desktop\Comp 755\Project
Output directory: C:\Users\14282\Desktop\Comp 755\Project\data\processed


## 2. Read both RDS files in Python

An RDS file normally contains one object. The helper below rejects ambiguous files rather than silently choosing an object.

In [3]:
def read_single_rds(path: Path) -> pd.DataFrame:
    objects = pyreadr.read_r(str(path))
    if len(objects) != 1:
        raise ValueError(f"Expected one object in {path.name}, found {len(objects)}")
    frame = next(iter(objects.values()))
    if not isinstance(frame, pd.DataFrame):
        raise TypeError(f"Expected a data frame in {path.name}, got {type(frame)!r}")
    return frame.reset_index(drop=True)

firm = read_single_rds(FIRM_RDS)
macro = read_single_rds(MACRO_RDS)

print(f"Firm panel: {firm.shape[0]:,} rows x {firm.shape[1]} columns")
print(f"Macro data: {macro.shape[0]:,} rows x {macro.shape[1]} columns")
display(firm.head(3))
display(macro.head(3))

Firm panel: 809,472 rows x 49 columns
Macro data: 180 rows x 10 columns


,A2ME,AC,AT,ATO,B2M,BETA_d,BETA_m,C2A,CF2B,CF2P,...,S2P,SGA2S,SPREAD,SUV,TURN,VAR,date,permno,Intervals,Early_Years
0,0.276440,-0.256877,0.368816,-0.295178,0.360955,0.30671,-0.029187,-0.424605,-0.156545,0.117530,...,-0.099198,-0.208049,-0.485418,0.398568,0.483073,-0.115114,19840331.0,10006.0,1984-1988,False
1,-0.292093,NaN,-0.232626,0.137317,-0.280899,NaN,NaN,0.264694,0.184898,0.005777,...,-0.048497,0.412945,0.122535,0.209001,0.057750,0.130027,19840331.0,10015.0,1984-1988,True
2,-0.303131,NaN,-0.308933,-0.032495,-0.196830,NaN,NaN,0.162546,0.238387,-0.124104,...,-0.106012,0.099293,0.306131,-0.410160,-0.372220,0.121649,19840331.0,10031.0,1984-1988,True


,Year,Quarter,dp,ep,bm,ntis,tbl,tms,dfy,svar
0,1977.0,1.0,-3.243842,-2.344591,0.827263,0.027896,0.0475,0.0274,0.0098,0.001118
1,1977.0,2.0,-3.181876,-2.298456,0.836804,0.027970,0.0467,0.0308,0.0108,0.000335
2,1977.0,3.0,-3.106215,-2.232803,0.888211,0.035315,0.0496,0.0269,0.0096,0.000635


## 3. Standardize Python dtypes and identify the 45 characteristics

The characteristic block is defined by the inclusive column range `A2ME:VAR`, matching the source dataset. The checks prevent accidental processing if the source schema changes.

In [4]:
required_firm_columns = {"date", "permno", "A2ME", "VAR", "Intervals", "Early_Years"}
required_macro_columns = {"Year", "Quarter", "dp", "ep", "bm", "ntis", "tbl", "tms", "dfy", "svar"}

missing_firm_columns = required_firm_columns.difference(firm.columns)
missing_macro_columns = required_macro_columns.difference(macro.columns)
assert not missing_firm_columns, f"Missing firm columns: {sorted(missing_firm_columns)}"
assert not missing_macro_columns, f"Missing macro columns: {sorted(missing_macro_columns)}"

start = firm.columns.get_loc("A2ME")
stop = firm.columns.get_loc("VAR")
assert start <= stop, "A2ME must occur before VAR in the source schema"
CHARACTERISTICS = firm.columns[start : stop + 1].tolist()
assert len(CHARACTERISTICS) == 45, f"Expected 45 characteristics, found {len(CHARACTERISTICS)}"

# Convert identifiers and dates to stable Python-friendly representations.
firm["date"] = pd.to_datetime(
    pd.to_numeric(firm["date"], errors="raise").astype("int64").astype(str),
    format="%Y%m%d",
    errors="raise",
)
firm["permno"] = pd.to_numeric(firm["permno"], errors="raise").astype("int64")
firm["Intervals"] = pd.Series(firm["Intervals"].astype(object), dtype="string")
firm["Early_Years"] = firm["Early_Years"].astype("boolean")
for column in CHARACTERISTICS:
    numeric_column = pd.to_numeric(firm[column], errors="raise")
    with np.errstate(invalid="ignore"):
        firm[column] = numeric_column.to_numpy(dtype=np.float32, na_value=np.nan)

macro["Year"] = pd.to_numeric(macro["Year"], errors="raise").astype("int16")
macro["Quarter"] = pd.to_numeric(macro["Quarter"], errors="raise").astype("int8")
macro_variables = [c for c in macro.columns if c not in {"Year", "Quarter"}]
for column in macro_variables:
    macro[column] = pd.to_numeric(macro[column], errors="raise").to_numpy(dtype=np.float32, na_value=np.nan)

assert not firm.duplicated(["date", "permno"]).any(), "Duplicate date-permno keys found"
assert not macro.duplicated(["Year", "Quarter"]).any(), "Duplicate Year-Quarter keys found"

observed = firm[CHARACTERISTICS].to_numpy(dtype=np.float32, copy=False)
observed_values = observed[~np.isnan(observed)]
assert observed_values.min() >= -0.5 and observed_values.max() <= 0.5, "Characteristic ranks fall outside [-0.5, 0.5]"

print(f"Characteristics ({len(CHARACTERISTICS)}): {CHARACTERISTICS}")
print(f"Date range: {firm['date'].min().date()} to {firm['date'].max().date()}")

Characteristics (45): ['A2ME', 'AC', 'AT', 'ATO', 'B2M', 'BETA_d', 'BETA_m', 'C2A', 'CF2B', 'CF2P', 'CTO', 'D2A', 'D2P', 'DPI2A', 'E2P', 'FC2Y', 'HIGH52', 'INV', 'IdioVol', 'LEV', 'ME', 'NI', 'NOA', 'OA', 'OL', 'OP', 'PCM', 'PM', 'PROF', 'Q', 'R12_2', 'R12_7', 'R2_1', 'R36_13', 'R60_13', 'RNA', 'ROA', 'ROE', 'RVAR', 'S2P', 'SGA2S', 'SPREAD', 'SUV', 'TURN', 'VAR']
Date range: 1984-03-31 to 2020-12-31


## 4. Export Python-native copies of the source data

After this cell, downstream Python work can use the Parquet files and does not need to read RDS again. The converted firm panel still contains the original missing values.

In [5]:
FIRM_PARQUET = OUTPUT_DIR / "firm_panel_python.parquet"
MACRO_PARQUET = OUTPUT_DIR / "macro_vars_python.parquet"

firm.to_parquet(FIRM_PARQUET, index=False, engine="pyarrow", compression="zstd")
macro.to_parquet(MACRO_PARQUET, index=False, engine="pyarrow", compression="zstd")

# Reopen the files to verify that the exported artifacts are readable in Python.
firm_check = pd.read_parquet(FIRM_PARQUET, columns=["date", "permno", *CHARACTERISTICS[:2]])
macro_check = pd.read_parquet(MACRO_PARQUET)
assert len(firm_check) == len(firm)
assert macro_check.shape == macro.shape
del firm_check, macro_check

print(f"Saved: {FIRM_PARQUET.name}")
print(f"Saved: {MACRO_PARQUET.name}")

Saved: firm_panel_python.parquet
Saved: macro_vars_python.parquet


## 5. Record missingness before constructing threshold bits

For characteristic $j$, the missingness bit is

$$M_{itj}=1\{X_{itj}\text{ is missing}\}.$$

No missing characteristic value is replaced before `missing_mask` is created.

In [6]:
values = firm[CHARACTERISTICS].to_numpy(dtype=np.float32, copy=True)
missing_mask = np.isnan(values)

source_missing_count = int(firm[CHARACTERISTICS].isna().to_numpy().sum())
assert int(missing_mask.sum()) == source_missing_count

missing_rates = pd.Series(missing_mask.mean(axis=0), index=CHARACTERISTICS, name="missing_rate")
print(f"Missing cells recorded: {source_missing_count:,} ({missing_mask.mean():.2%})")
display(missing_rates.sort_values(ascending=False).head(10).to_frame())

Missing cells recorded: 3,736,252 (10.26%)


,missing_rate
DPI2A,0.490013
R60_13,0.357143
HIGH52,0.342774
AC,0.268752
R36_13,0.229768
BETA_d,0.222035
OA,0.217767
SGA2S,0.212345
FC2Y,0.212325
D2A,0.156832


## 6. Create the 225-dimensional binary matrix

For each of the 45 characteristics, create four ordered rank-threshold bits at $(-0.3,-0.1,0.1,0.3)$ and one missingness bit:

$$B_{itjr}=1\{M_{itj}=0,\ X_{itj}>\tau_r\},\qquad r=1,\ldots,4.$$

All threshold bits are zero when a characteristic is missing. Each source characteristic therefore contributes five bits, for a total dimension of $45\times5=225$.

In [7]:
THRESHOLDS = np.array([-0.3, -0.1, 0.1, 0.3], dtype=np.float32)
BITS_PER_CHARACTERISTIC = len(THRESHOLDS) + 1
N_ROWS = len(firm)
N_BINARY_FEATURES = len(CHARACTERISTICS) * BITS_PER_CHARACTERISTIC

X_binary = np.zeros((N_ROWS, N_BINARY_FEATURES), dtype=np.uint8)
binary_feature_names = []
feature_schema = {}

def threshold_label(value: float) -> str:
    sign = "m" if value < 0 else "p"
    magnitude = f"{abs(value):.1f}".replace(".", "p")
    return f"gt_{sign}{magnitude}"

for j, characteristic in enumerate(CHARACTERISTICS):
    base = j * BITS_PER_CHARACTERISTIC
    observed_j = ~missing_mask[:, j]
    names_j = []

    for r, threshold in enumerate(THRESHOLDS):
        # The observed_j condition guarantees zero threshold bits for missing values.
        X_binary[:, base + r] = (observed_j & (values[:, j] > threshold)).astype(np.uint8)
        names_j.append(f"{characteristic}__{threshold_label(float(threshold))}")

    X_binary[:, base + len(THRESHOLDS)] = missing_mask[:, j].astype(np.uint8)
    names_j.append(f"{characteristic}__missing")
    binary_feature_names.extend(names_j)
    feature_schema[characteristic] = {
        "column_indices": list(range(base, base + BITS_PER_CHARACTERISTIC)),
        "binary_features": names_j,
    }

assert X_binary.shape == (N_ROWS, 225)
assert len(binary_feature_names) == 225
print(f"Binary matrix: {X_binary.shape[0]:,} rows x {X_binary.shape[1]} columns, dtype={X_binary.dtype}")
print(f"Memory size: {X_binary.nbytes / 1024**2:.1f} MiB")

Binary matrix: 809,472 rows x 225 columns, dtype=uint8
Memory size: 173.7 MiB


## 7. Validate binary values, missingness preservation, and ordered encoding

In [8]:
# Global binary and shape checks.
assert X_binary.dtype == np.uint8
assert np.logical_or(X_binary == 0, X_binary == 1).all(), "Non-binary values found"

threshold_columns = np.concatenate([
    np.arange(j * BITS_PER_CHARACTERISTIC, j * BITS_PER_CHARACTERISTIC + len(THRESHOLDS))
    for j in range(len(CHARACTERISTICS))
])
missing_columns = np.arange(
    len(THRESHOLDS), N_BINARY_FEATURES, BITS_PER_CHARACTERISTIC
)

# Missing indicators reproduce the original NA mask exactly.
encoded_missing = X_binary[:, missing_columns]
assert np.array_equal(encoded_missing, missing_mask.astype(np.uint8))
assert int(encoded_missing.sum()) == source_missing_count

# For every characteristic, missing rows have zero threshold bits and observed bits are ordered.
for j, characteristic in enumerate(CHARACTERISTICS):
    base = j * BITS_PER_CHARACTERISTIC
    bits = X_binary[:, base : base + len(THRESHOLDS)]
    miss = X_binary[:, base + len(THRESHOLDS)].astype(bool)
    assert not bits[miss].any(), f"Nonzero threshold bit found for missing {characteristic}"
    assert np.all(bits[:, :-1] >= bits[:, 1:]), f"Non-monotone threshold encoding for {characteristic}"

bit_prevalence = pd.Series(X_binary.mean(axis=0), index=binary_feature_names, name="prevalence")
constant_binary_features = bit_prevalence[(bit_prevalence == 0) | (bit_prevalence == 1)].index.tolist()
constant_threshold_features = [name for name in constant_binary_features if not name.endswith("__missing")]
assert not constant_threshold_features, f"Constant threshold features found: {constant_threshold_features}"

print("All structural validation checks passed.")
if constant_binary_features:
    print("Constant missingness indicators retained for a stable 225-column schema:")
    print(constant_binary_features)
display(bit_prevalence.describe().to_frame())

All structural validation checks passed.
Constant missingness indicators retained for a stable 225-column schema:
['D2P__missing', 'ME__missing', 'R2_1__missing', 'SPREAD__missing']


,prevalence
count,225.000000
mean,0.379792
std,0.238377
min,0.000000
25%,0.185770
50%,0.373672
75%,0.575180
max,0.813555


## 8. Save the model matrix, row index, feature names, and metadata

The `.npy` matrix can be memory-mapped during model training. `binary_rows.parquet` preserves the row-to-firm/date mapping without placing identifiers inside the likelihood model.

In [9]:
BINARY_NPY = OUTPUT_DIR / "binary_features_225.npy"
ROW_INDEX_PARQUET = OUTPUT_DIR / "binary_rows.parquet"
FEATURE_NAMES_JSON = OUTPUT_DIR / "binary_feature_names.json"
METADATA_JSON = OUTPUT_DIR / "binary_feature_metadata.json"
PREVALENCE_PARQUET = OUTPUT_DIR / "binary_feature_prevalence.parquet"

np.save(BINARY_NPY, X_binary, allow_pickle=False)

row_index = firm[["date", "permno", "Intervals", "Early_Years"]].copy()
row_index.insert(0, "row_id", np.arange(len(row_index), dtype=np.int64))
row_index.to_parquet(ROW_INDEX_PARQUET, index=False, engine="pyarrow", compression="zstd")

with FEATURE_NAMES_JSON.open("w", encoding="utf-8") as file:
    json.dump(binary_feature_names, file, indent=2)

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

metadata = {
    "source_files": {
        FIRM_RDS.name: {"sha256": sha256_file(FIRM_RDS)},
        MACRO_RDS.name: {"sha256": sha256_file(MACRO_RDS)},
    },
    "n_rows": int(N_ROWS),
    "n_source_characteristics": len(CHARACTERISTICS),
    "n_binary_features": int(N_BINARY_FEATURES),
    "dtype": str(X_binary.dtype),
    "thresholds": [float(x) for x in THRESHOLDS],
    "missing_encoding": {
        "threshold_bits_when_missing": 0,
        "missing_bit_when_missing": 1,
    },
    "characteristics": CHARACTERISTICS,
    "feature_schema": feature_schema,
    "date_range": [str(firm["date"].min().date()), str(firm["date"].max().date())],
    "source_missing_cells": source_missing_count,
    "constant_binary_features": constant_binary_features,
}
with METADATA_JSON.open("w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

bit_prevalence.rename_axis("feature").reset_index().to_parquet(
    PREVALENCE_PARQUET, index=False, engine="pyarrow", compression="zstd"
)

print("Saved model-ready artifacts:")
for path in [BINARY_NPY, ROW_INDEX_PARQUET, FEATURE_NAMES_JSON, METADATA_JSON, PREVALENCE_PARQUET]:
    print(f"  {path.name}: {path.stat().st_size / 1024**2:.2f} MiB")

Saved model-ready artifacts:
  binary_features_225.npy: 173.69 MiB
  binary_rows.parquet: 2.30 MiB
  binary_feature_names.json: 0.00 MiB
  binary_feature_metadata.json: 0.01 MiB
  binary_feature_prevalence.parquet: 0.00 MiB


## 9. Reopen the saved artifacts

This final check demonstrates how subsequent modeling notebooks should load the data without R.

In [10]:
X_loaded = np.load(BINARY_NPY, mmap_mode="r", allow_pickle=False)
rows_loaded = pd.read_parquet(ROW_INDEX_PARQUET)
macro_loaded = pd.read_parquet(MACRO_PARQUET)
with FEATURE_NAMES_JSON.open(encoding="utf-8") as file:
    names_loaded = json.load(file)

assert X_loaded.shape == (len(rows_loaded), len(names_loaded)) == (N_ROWS, 225)
assert X_loaded.dtype == np.uint8
assert macro_loaded.shape == macro.shape

print("Python-only reload check passed.")
display(rows_loaded.head(3))
display(pd.DataFrame(np.asarray(X_loaded[:3, :15]), columns=names_loaded[:15]))

Python-only reload check passed.


,row_id,date,permno,Intervals,Early_Years
0,0,1984-03-31,10006,1984-1988,False
1,1,1984-03-31,10015,1984-1988,True
2,2,1984-03-31,10031,1984-1988,True


,A2ME__gt_m0p3,A2ME__gt_m0p1,A2ME__gt_p0p1,A2ME__gt_p0p3,A2ME__missing,AC__gt_m0p3,AC__gt_m0p1,AC__gt_p0p1,AC__gt_p0p3,AC__missing,AT__gt_m0p3,AT__gt_m0p1,AT__gt_p0p1,AT__gt_p0p3,AT__missing
0,1,1,1,0,0,1,0,0,0,0,1,1,1,1,0
1,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0
2,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0


## Output manifest

The notebook creates these files under `data/processed/`:

- `firm_panel_python.parquet` — Python-native copy of the firm panel, retaining missing values;
- `macro_vars_python.parquet` — Python-native copy of the macro data;
- `binary_features_225.npy` — model-ready 225-dimensional `uint8` matrix;
- `binary_rows.parquet` — row IDs, dates, PERMNOs, intervals, and `Early_Years`;
- `binary_feature_names.json` — ordered matrix column names;
- `binary_feature_metadata.json` — thresholds, schema, dimensions, missing convention, and source hashes;
- `binary_feature_prevalence.parquet` — prevalence diagnostic for every generated bit.